In [3]:
#  Import libraries
import pandas as pd
import pickle
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error

# 1. Load dataset
print("Loading California Housing dataset...")
X, y = fetch_california_housing(return_X_y=True, as_frame=True)

# Explore the dataset
print(f"Dataset shape: {X.shape}")
display(X.head())

Loading California Housing dataset...
Dataset shape: (20640, 8)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [4]:
# 2. Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Preprocessing: Imputation + Scaling for numerical features
numeric_features = X.columns  # All 8 features are numerical
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# 4. Combine preprocessing using ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features)
])

# 5. Build full pipeline: preprocessing + KNN Regressor
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('knn', KNeighborsRegressor())
])

print("Pipeline constructed successfully!")

Pipeline constructed successfully!


In [5]:
# 6. Define hyperparameter grid
param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2] # 1: Manhattan distance, 2: Euclidean distance
}

# 7. Apply GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    verbose=1,
    n_jobs=-1
)

# 8. Fit the model (This might take a minute depending on Colab resources)
print("Starting Grid Search...")
grid_search.fit(X_train, y_train)
print("Grid Search complete!")

Starting Grid Search...
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Grid Search complete!


In [7]:
# 9. Evaluate on test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# Calculate metrics
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5 # Updated to avoid deprecation warnings

# 10. Print results
print("\n--- Model Evaluation ---")
print("Best Parameters:", grid_search.best_params_)
print("Best CV R² Score:", round(grid_search.best_score_, 4))
print("Test R² Score:", round(r2, 4))
print("Test MSE:", round(mse, 4))
print("Test RMSE:", round(rmse, 4))

# 11. Save the pipeline
filename = 'california_knn_pipeline.pkl'
with open(filename, 'wb') as f:
    pickle.dump(best_model, f)

print(f"\n Final pipeline saved successfully to '{filename}'")


--- Model Evaluation ---
Best Parameters: {'knn__n_neighbors': 9, 'knn__p': 1, 'knn__weights': 'distance'}
Best CV R² Score: 0.7313
Test R² Score: 0.7221
Test MSE: 0.3642
Test RMSE: 0.6034

 Final pipeline saved successfully to 'california_knn_pipeline.pkl'
